# Multi-Modal Image Forgery Detection: Visual Artifacts & Metadata Fusion


## 1. Environment Setup
Importing all required libraries for deep learning, gradient boosting, statistical analysis, and visualisation.


In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, ExifTags
from tqdm.notebook import tqdm

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, ResNet50, VGG16, EfficientNetB4
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# XGBoost
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams.update({'figure.figsize': (10, 6)})
print(f"TensorFlow: {tf.__version__} | XGBoost: {xgb.__version__}")

## 2. Configuration
Centralised hyperparameters, dataset paths, and reproducibility settings.


In [ ]:
class Config:
    SEED       = 42
    IMG_SIZE   = (224, 224)
    BATCH_SIZE = 32
    EPOCHS     = 5

    # Kaggle dataset paths
    PATH_CASIA1  = "./data/CASIA1/CASIA1"
    PATH_CASIA2  = "./data/CASIA2/CASIA2"
    PATH_COMOFOD = "./data/CoMoFoD_small_v2/CoMoFoD_small_v2"
    PATH_CG1050  = "./data/CG-1050/CG-1050"

    #PATH_CASIA1  = " "
    #PATH_CASIA2  = " "
    #PATH_COMOFOD = " "
    #PATH_CG1050  = " "

    OUTPUT_BASE = "./outputs"
    MODEL_DIR   = os.path.join(OUTPUT_BASE, "models")
    FIGURE_DIR  = os.path.join(OUTPUT_BASE, "figures")

    @staticmethod
    def setup():
        Config.set_seed()
        os.makedirs(Config.MODEL_DIR,  exist_ok=True)
        os.makedirs(Config.FIGURE_DIR, exist_ok=True)
        print(f"Output directories ready.")

    @staticmethod
    def set_seed():
        np.random.seed(Config.SEED)
        tf.random.set_seed(Config.SEED)
        os.environ['PYTHONHASHSEED'] = str(Config.SEED)
        print(f"Seed set: {Config.SEED}")

Config.setup()

## 3. Data Utility Functions
Helpers to recursively locate valid images and verify file integrity before loading.


In [ ]:
def get_valid_image_paths(directory, exts={".jpg",".jpeg",".png",".tif",".tiff",".bmp"}):
    """Recursively find all valid image paths in a directory."""
    valid = []
    for root, _, files in os.walk(directory):
        for f in files:
            if os.path.splitext(f)[1].lower() in exts:
                valid.append(os.path.join(root, f))
    return valid

def check_image_validity(path):
    """Verify an image file is not corrupt."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except:
        return False

## 4. Dataset Loading
Parsers for the directory structures of CASIA, CoMoFoD, and CG-1050 datasets.


In [ ]:
def load_casia(base_path, dataset_name):
    """Load CASIA dataset: Au=Authentic (0), Tp=Manipulated (1)."""
    data = []
    if not os.path.exists(base_path):
        print(f"Not found: {base_path}"); return pd.DataFrame()
    for f in get_valid_image_paths(os.path.join(base_path, 'Au')):
        data.append({'path': f, 'label': 0, 'dataset': dataset_name, 'class': 'Authentic'})
    for f in get_valid_image_paths(os.path.join(base_path, 'Tp')):
        data.append({'path': f, 'label': 1, 'dataset': dataset_name, 'class': 'Manipulated'})
    df = pd.DataFrame(data)
    print(f"{dataset_name}: {len(df)} images  "
          f"(Authentic={len(df[df.label==0])}, Manipulated={len(df[df.label==1])})")
    return df

def load_comofod(base_path):
    """Load CoMoFoD dataset."""
    data = []
    if not os.path.exists(base_path):
        print(f"Not found: {base_path}"); return pd.DataFrame()
    for f in get_valid_image_paths(os.path.join(base_path, 'Au')):
        data.append({'path': f, 'label': 0, 'dataset': 'CoMoFoD', 'class': 'Authentic'})
    for f in get_valid_image_paths(os.path.join(base_path, 'Tp')):
        data.append({'path': f, 'label': 1, 'dataset': 'CoMoFoD', 'class': 'Manipulated'})
    df = pd.DataFrame(data)
    print(f"CoMoFoD: {len(df)} images  "
          f"(Authentic={len(df[df.label==0])}, Manipulated={len(df[df.label==1])})")
    return df

def load_cg1050(base_path):
    """Load CG-1050 dataset."""
    data = []
    if not os.path.exists(base_path):
        print(f"Not found: {base_path}"); return pd.DataFrame()
    for f in get_valid_image_paths(os.path.join(base_path, 'Au')):
        data.append({'path': f, 'label': 0, 'dataset': 'CG1050', 'class': 'Authentic'})
    for f in get_valid_image_paths(os.path.join(base_path, 'Tp')):
        data.append({'path': f, 'label': 1, 'dataset': 'CG1050', 'class': 'Manipulated'})
    df = pd.DataFrame(data)
    print(f"CG-1050: {len(df)} images  "
          f"(Authentic={len(df[df.label==0])}, Manipulated={len(df[df.label==1])})")
    return df

## 5. Unified Dataset Index
Aggregating all four datasets into a single DataFrame with deduplication.


In [ ]:
df_casia1  = load_casia(Config.PATH_CASIA1,  'CASIA1')
df_casia2  = load_casia(Config.PATH_CASIA2,  'CASIA2')
df_comofod = load_comofod(Config.PATH_COMOFOD)
df_cg1050  = load_cg1050(Config.PATH_CG1050)

df = pd.concat([df_casia1, df_casia2, df_comofod, df_cg1050], ignore_index=True)
df = df.drop_duplicates(subset=['path']).reset_index(drop=True)

print("=" * 62)
print(f"{'Dataset':<12} | {'Authentic':>12} | {'Manipulated':>12} | {'Total':>8}")
print("-" * 62)
total_a = total_m = 0
for ds in sorted(df['dataset'].unique()):
    s = df[df['dataset'] == ds]
    a, m = len(s[s.label==0]), len(s[s.label==1])
    total_a += a; total_m += m
    print(f"{ds:<12} | {a:>12} | {m:>12} | {len(s):>8}")
print("-" * 62)
print(f"{'Combined':<12} | {total_a:>12} | {total_m:>12} | {len(df):>8}")
print("=" * 62)

## 6. Stratified Data Splitting
Stratified 80/20 train–validation split applied independently per dataset to preserve class balance.


In [ ]:
def create_stratified_splits(df):
    train_dfs, val_dfs = [], []
    for dataset in df['dataset'].unique():
        sub = df[df['dataset'] == dataset]
        tr, vl = train_test_split(sub, test_size=0.2,
                                  stratify=sub['label'], random_state=Config.SEED)
        train_dfs.append(tr); val_dfs.append(vl)
    train_df = pd.concat(train_dfs, ignore_index=True)
    val_df   = pd.concat(val_dfs,   ignore_index=True)
    df.loc[df['path'].isin(train_df['path']), 'split'] = 'TRAIN'
    df.loc[df['path'].isin(val_df['path']),   'split'] = 'VALIDATION'
    print(f"Training set:   {len(train_df):,} images")
    print(f"Validation set: {len(val_df):,} images")
    return df

df = create_stratified_splits(df)
df.to_csv('dataset_index.csv', index=False)
print("Saved: dataset_index.csv")

## 7. Visual Branch: MobileNetV2 Architecture
Pre-trained MobileNetV2 backbone with a custom forensic classification head. Backbone weights are frozen; only the projection head is trained.


In [ ]:
def build_visual_model(input_shape=Config.IMG_SIZE + (3,)):
    """MobileNetV2 backbone (frozen) + 128-D projection head."""
    base = MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = False

    inputs  = Input(shape=input_shape)
    x       = base(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    features= Dense(128, activation='relu', name='visual_features')(x)
    x       = Dropout(0.5)(features)
    outputs = Dense(1, activation='sigmoid', name='visual_output')(x)
    model   = Model(inputs, outputs, name='Visual_MobileNetV2')
    return model

visual_model = build_visual_model()
visual_model.summary()
visual_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

## 8. Data Generators
Keras ImageDataGenerators with augmentation for training and normalisation-only for validation.


In [ ]:
def create_generators(df, img_size=Config.IMG_SIZE):
    train_df = df[df['split'] == 'TRAIN']
    val_df   = df[df['split'] == 'VALIDATION']

    aug_gen  = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1./255, rotation_range=20, width_shift_range=0.2,
        height_shift_range=0.2, horizontal_flip=True, fill_mode='nearest')
    val_gen_ = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

    train_g = aug_gen.flow_from_dataframe(
        train_df, x_col='path', y_col='label',
        target_size=img_size, batch_size=Config.BATCH_SIZE,
        class_mode='raw', shuffle=True)
    val_g = val_gen_.flow_from_dataframe(
        val_df, x_col='path', y_col='label',
        target_size=img_size, batch_size=Config.BATCH_SIZE,
        class_mode='raw', shuffle=False)
    return train_g, val_g

train_gen, val_gen = create_generators(df)

## 9. Training the Visual Model
Training with Early Stopping and Learning Rate Reduction callbacks. Best model saved by validation AUC.


In [ ]:
model_path = os.path.join(Config.MODEL_DIR, 'visual_model_best.h5')

callbacks = [
    ModelCheckpoint(model_path, save_best_only=True, monitor='val_auc', mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("Training Visual Model...")
history_visual = visual_model.fit(
    train_gen, validation_data=val_gen,
    epochs=Config.EPOCHS, callbacks=callbacks
)

## 10. Evaluating the Visual Model
Training curves, confusion matrix, and classification report for the MobileNetV2 baseline.


In [ ]:
def plot_training_history(history, prefix=""):
    metrics = ['loss', 'accuracy', 'auc']
    plt.figure(figsize=(18, 5))
    for i, m in enumerate(metrics):
        plt.subplot(1, 3, i+1)
        plt.plot(history.history[m], label=f'Train {m}')
        plt.plot(history.history[f'val_{m}'], label=f'Val {m}')
        plt.title(f'{prefix} {m.capitalize()}'); plt.legend(); plt.grid(True)
    path = os.path.join(Config.FIGURE_DIR, f"{prefix.replace(' ','_')}_training_history.png")
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history_visual, "Visual Model")

# Ordered generator (no shuffle) for evaluation
_, val_gen_ordered = create_generators(df)

y_prob_visual = visual_model.predict(val_gen_ordered).ravel()
y_pred_visual = (y_prob_visual > 0.5).astype(int)
y_true_vis    = val_gen_ordered.labels

cm = confusion_matrix(y_true_vis, y_pred_visual)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic','Manipulated'],
            yticklabels=['Authentic','Manipulated'])
plt.title('Visual Model — Confusion Matrix')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.savefig(os.path.join(Config.FIGURE_DIR,'Visual_Confusion_Matrix.png'), dpi=300)
plt.show()
print(classification_report(y_true_vis, y_pred_visual,
                             target_names=['Authentic','Manipulated']))

## 11. Metadata Feature Extraction
Extracting 15 metadata attributes per image: 14 EXIF tags and 1 file system attribute.
Categorical fields are label-encoded; missing tags are represented as −1 (treated as
an informative sentinel by XGBoost, since absence of metadata is itself a forensic signal).


In [ ]:
EXIF_FIELDS = ['Make','Model','Software','DateTime',
               'ExifImageWidth','ExifImageHeight','ExposureProgram',
               'ISOSpeedRatings','ShutterSpeedValue','ApertureValue',
               'MeteringMode','Flash','FocalLength','ColorSpace']

def extract_exif(path):
    try:
        img = Image.open(path)
        raw = img._getexif()
        return {ExifTags.TAGS.get(k,k): v for k,v in raw.items()} if raw else {}
    except:
        return {}

def process_metadata(df):
    """Extract and encode metadata features for all images."""
    print("Extracting EXIF metadata...")
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        exif = extract_exif(row['path'])
        feat = {k: str(exif.get(k, 'Unknown')) for k in EXIF_FIELDS}
        try:    feat['FileSize'] = os.stat(row['path']).st_size
        except: feat['FileSize'] = 0
        records.append(feat)

    meta_df = pd.DataFrame(records)
    print("Label-encoding categorical fields...")
    for col in meta_df.columns:
        if meta_df[col].dtype == 'object':
            meta_df[col] = LabelEncoder().fit_transform(meta_df[col].astype(str))
    return meta_df.astype(float)

META_CSV = 'metadata_features.csv'
X_meta = process_metadata(df)
X_meta.to_csv(META_CSV, index=False)
print(f"Saved {META_CSV}  |  Shape: {X_meta.shape}")

## 12. Metadata Model Training (XGBoost)
Training a standalone XGBoost classifier on EXIF features, with detailed evaluation.


In [ ]:
y_train = df[df['split'] == 'TRAIN']['label'].reset_index(drop=True)
y_val   = df[df['split'] == 'VALIDATION']['label'].reset_index(drop=True)
X_meta_train = X_meta[df['split'] == 'TRAIN'].reset_index(drop=True)
X_meta_val   = X_meta[df['split'] == 'VALIDATION'].reset_index(drop=True)

xgb_meta = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5,
    use_label_encoder=False, eval_metric='logloss', random_state=Config.SEED
)
print("Training metadata-only XGBoost classifier...")
xgb_meta.fit(X_meta_train, y_train,
             eval_set=[(X_meta_val, y_val)], verbose=False)

y_prob_meta = xgb_meta.predict_proba(X_meta_val)[:, 1]
y_pred_meta = (y_prob_meta > 0.5).astype(int)
print(f"Metadata — Accuracy: {accuracy_score(y_val, y_pred_meta)*100:.2f}%  "
      f"AUC: {roc_auc_score(y_val, y_prob_meta):.4f}")

cm_meta = confusion_matrix(y_val, y_pred_meta)
plt.figure(figsize=(6,5))
sns.heatmap(cm_meta, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Authentic','Manipulated'],
            yticklabels=['Authentic','Manipulated'])
plt.title('Metadata Model — Confusion Matrix')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.savefig(os.path.join(Config.FIGURE_DIR,'Metadata_Confusion_Matrix.png'), dpi=300)
plt.show()
print(classification_report(y_val, y_pred_meta,
                             target_names=['Authentic','Manipulated']))

## 13. Feature Importance - Forensic Interpretation
Ranking EXIF features by XGBoost gain score and annotating each with its forensic significance to explain *why* a feature is discriminative (not just *that* it is).


In [ ]:
feature_names = EXIF_FIELDS + ['FileSize']

forensic_notes = {
    'Software':        'Editing software signature (e.g. Photoshop, GIMP)',
    'DateTime':        'Re-save timestamp inconsistency',
    'Make':            'Camera brand absent / generic in edited files',
    'Model':           'Camera model absent / mismatched',
    'FileSize':        'File size altered by re-compression',
    'ISOSpeedRatings': 'Physically implausible values in CGI/synthetic files',
    'FocalLength':     'Absent in screen captures and AI-generated images',
    'ColorSpace':      'Altered by editing tools (sRGB <-> Adobe RGB)',
    'ExposureProgram': 'Default values reset by editing software',
    'MeteringMode':    'Default values reset by editing software',
    'Flash':           'Inconsistent with scene lighting conditions',
    'ShutterSpeedValue':'Implausible value range for claimed exposure',
    'ApertureValue':   'Cross-check with shutter speed for plausibility',
    'ExifImageWidth':  'Declared width mismatches actual after cropping',
    'ExifImageHeight': 'Declared height mismatches actual after cropping',
}

importance_scores = xgb_meta.feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_names,
                          'Importance': importance_scores}
                        ).sort_values('Importance', ascending=False)

top10 = feat_imp.head(10)
colors_imp = ['#C44E52' if i < 3 else '#4C72B0' for i in range(len(top10))]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(top10['Feature'][::-1], top10['Importance'][::-1],
               color=colors_imp[::-1], edgecolor='white')
for bar, feat in zip(bars, top10['Feature'][::-1]):
    note = forensic_notes.get(feat, '')
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
            f'  ← {note}', va='center', fontsize=8.5, color='#333')

ax.set_xlabel('Feature Importance (XGBoost Gain)', fontsize=11)
ax.set_title('Top Metadata Features — Forensic Interpretation\n'
             '(Red bars = strongest discriminators)', fontsize=12, fontweight='bold')
ax.xaxis.grid(True, linestyle='--', alpha=0.6); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'Feature_Importance_Annotated.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print(feat_imp.head(10).to_string(index=False))

## 14. Visual Feature Extraction for Fusion
Extracting 128-dimensional embeddings from the trained MobileNetV2 projection head for every image.


In [ ]:
def extract_visual_features(model, generator):
    """Extract 128-D embeddings from the visual_features layer."""
    extractor = Model(inputs=model.input,
                      outputs=model.get_layer('visual_features').output)
    print(f"Extracting features for {len(generator.filenames)} images...")
    return extractor.predict(generator, verbose=1)

# Ordered generators (shuffle=False) to align with metadata indices
train_gen_ord, val_gen_ord = create_generators(df)
train_gen_ord.shuffle = False

X_visual_train = extract_visual_features(visual_model, train_gen_ord)
X_visual_val   = extract_visual_features(visual_model, val_gen_ord)
print(f"Visual feature shapes — Train: {X_visual_train.shape} | Val: {X_visual_val.shape}")

## 15. Feature Concatenation
Combining MobileNetV2 visual embeddings (128-D) with EXIF metadata vectors (15-D) into a joint 143-D multimodal feature set.


In [ ]:
X_meta_train_aligned = X_meta[df['split'] == 'TRAIN'].reset_index(drop=True).values
X_meta_val_aligned   = X_meta[df['split'] == 'VALIDATION'].reset_index(drop=True).values

X_fusion_train = np.hstack([X_visual_train, X_meta_train_aligned])
X_fusion_val   = np.hstack([X_visual_val,   X_meta_val_aligned])

print(f"Fusion feature shape — Train: {X_fusion_train.shape}")
print(f"  Visual dims: {X_visual_train.shape[1]}  |  Metadata dims: {X_meta_train_aligned.shape[1]}")

## 16. Fusion Model Training
Training the hybrid XGBoost classifier on the concatenated 143-D feature space.


In [ ]:
xgb_fusion = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss', random_state=Config.SEED
)
print("Training hybrid fusion XGBoost classifier...")
xgb_fusion.fit(X_fusion_train, y_train,
               eval_set=[(X_fusion_val, y_val)], verbose=False)

y_prob_fusion = xgb_fusion.predict_proba(X_fusion_val)[:, 1]
y_pred_fusion = (y_prob_fusion > 0.5).astype(int)
print(f"Fusion — Accuracy: {accuracy_score(y_val, y_pred_fusion)*100:.2f}%  "
      f"AUC: {roc_auc_score(y_val, y_prob_fusion):.4f}")

## 17. Final Comparative Evaluation
Comparing Visual, Metadata, and Fusion models using ROC curves, confusion matrices, and full classification reports.


In [ ]:
y_true_np = y_val.values

# ── ROC Curve Comparison ─────────────────────────────────────
plt.figure(figsize=(9, 7))
for name, y_prob in [('Visual (MobileNetV2)', y_prob_visual),
                      ('Metadata (XGBoost)',    y_prob_meta),
                      ('Fusion (Proposed)',      y_prob_fusion)]:
    fpr, tpr, _ = roc_curve(y_true_np, y_prob)
    auc = roc_auc_score(y_true_np, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.4f})')
plt.plot([0,1],[0,1],'k--',alpha=0.4)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Visual vs. Metadata vs. Fusion', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(Config.FIGURE_DIR,'ROC_Comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── Fusion Confusion Matrix ───────────────────────────────────
cm_fus = confusion_matrix(y_true_np, y_pred_fusion)
plt.figure(figsize=(6,5))
sns.heatmap(cm_fus, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Authentic','Manipulated'],
            yticklabels=['Authentic','Manipulated'])
plt.title('Fusion Model — Confusion Matrix')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.savefig(os.path.join(Config.FIGURE_DIR,'Fusion_Confusion_Matrix.png'), dpi=300)
plt.show()

print("Fusion Model Classification Report:")
print(classification_report(y_true_np, y_pred_fusion,
                             target_names=['Authentic','Manipulated']))

# ── Per-dataset accuracy breakdown ───────────────────────────
results_df = df[df['split'] == 'VALIDATION'].copy().reset_index(drop=True)
results_df['Visual_Prob']  = y_prob_visual
results_df['Meta_Prob']    = y_prob_meta
results_df['Fusion_Prob']  = y_prob_fusion
results_df['Fusion_Pred']  = y_pred_fusion

print("\nPer-Dataset Accuracy (Fusion Model):")
print("-" * 42)
for ds in sorted(results_df['dataset'].unique()):
    sub = results_df[results_df['dataset'] == ds]
    acc = accuracy_score(sub['label'], sub['Fusion_Pred'])
    print(f"  {ds:<10}: {acc*100:.2f}%")

## 18. Ablation Study
Systematic comparison of four fusion configurations to isolate the contribution of each pipeline component: visual features, metadata features, score-level averaging, and the proposed feature-level concatenation.


In [ ]:
ablation_configs = {}

# A: Visual stream only
y_prob_vis_abl = y_prob_visual
y_pred_vis_abl = (y_prob_vis_abl > 0.5).astype(int)
ablation_configs['A — Visual Only'] = (y_pred_vis_abl, y_prob_vis_abl)

# B: Metadata stream only
y_pred_met_abl = y_pred_meta
y_prob_met_abl = y_prob_meta
ablation_configs['B — Metadata Only'] = (y_pred_met_abl, y_prob_met_abl)

# C: Simple average fusion (probability-level, no learned interaction)
y_prob_avg = (y_prob_vis_abl + y_prob_met_abl) / 2.0
y_pred_avg = (y_prob_avg > 0.5).astype(int)
ablation_configs['C — Average Fusion'] = (y_pred_avg, y_prob_avg)

# D: Proposed feature-level fusion
y_pred_fus_abl = y_pred_fusion
y_prob_fus_abl = y_prob_fusion
ablation_configs['D — Feature-Level Fusion (Proposed)'] = (y_pred_fus_abl, y_prob_fus_abl)

abl_rows = []
for cfg_name, (y_pred_, y_prob_) in ablation_configs.items():
    abl_rows.append({
        'Configuration': cfg_name,
        'Accuracy (%)':  round(accuracy_score(y_true_np, y_pred_)*100, 2),
        'Precision (%)': round(precision_score(y_true_np, y_pred_, zero_division=0)*100, 2),
        'Recall (%)':    round(recall_score(y_true_np, y_pred_, zero_division=0)*100, 2),
        'F1 (%)':        round(f1_score(y_true_np, y_pred_, zero_division=0)*100, 2),
        'AUC-ROC':       round(roc_auc_score(y_true_np, y_prob_), 4),
    })

abl_df = pd.DataFrame(abl_rows)
print("\nAblation Study Results:")
print(abl_df.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(4)
w = 0.16
cmap = ['#4C72B0','#DD8452','#55A868','#C44E52']
for i, (metric, color) in enumerate(zip(['Accuracy (%)','Precision (%)','Recall (%)','F1 (%)'], cmap)):
    vals = abl_df[metric].values
    bars = ax.bar(x + i*w, vals, w, label=metric, color=color, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(['A: Visual\nOnly','B: Metadata\nOnly',
                    'C: Average\nFusion','D: Feature-Level\nFusion'], fontsize=10)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_ylim(0, 112)
ax.set_title('Ablation Study — Fusion Configuration Comparison', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.6); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'Ablation_Study.png'), dpi=300, bbox_inches='tight')
plt.show()

## 19. Modality Contribution Analysis
Quantifying the relative influence of the visual and metadata streams via XGBoost feature gain, per-dataset confidence distributions (KDE), and modality dominance analysis.


In [ ]:
n_vis  = X_visual_val.shape[1]   # 128
n_meta = X_meta_val_aligned.shape[1]  # 15

imp = xgb_fusion.get_booster().get_score(importance_type='gain')
vis_gain  = sum(v for k,v in imp.items() if int(k.replace('f','')) < n_vis)
meta_gain = sum(v for k,v in imp.items() if int(k.replace('f','')) >= n_vis)
total     = vis_gain + meta_gain
vis_pct   = 100 * vis_gain  / total
meta_pct  = 100 * meta_gain / total

print(f"Visual stream contribution:   {vis_pct:.1f}%")
print(f"Metadata stream contribution: {meta_pct:.1f}%")

val_analysis = df[df['split']=='VALIDATION'].copy().reset_index(drop=True)
val_analysis['Visual_Prob']     = y_prob_visual
val_analysis['Meta_Prob']       = y_prob_meta
val_analysis['Visual_Dominant'] = (y_prob_visual > y_prob_meta).astype(int)

# ── Panel 1 & 2: Pie + Stacked bar ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie([vis_pct, meta_pct],
            labels=[f'Visual Stream\n({vis_pct:.1f}%)',
                    f'Metadata Stream\n({meta_pct:.1f}%)'],
            colors=['#4C72B0','#DD8452'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Overall Feature Gain\nby Modality', fontsize=13, fontweight='bold')

datasets_mc = sorted(val_analysis['dataset'].unique())
dom_rows = []
for ds in datasets_mc:
    sub = val_analysis[val_analysis['dataset']==ds]
    vd  = sub['Visual_Dominant'].mean()*100
    dom_rows.append({'Dataset': ds, 'Visual (%)': vd, 'Metadata (%)': 100-vd})
dom_df = pd.DataFrame(dom_rows).set_index('Dataset')
dom_df.plot(kind='bar', stacked=True, ax=axes[1],
            color=['#4C72B0','#DD8452'], width=0.55, edgecolor='white')
axes[1].set_title('Per-Dataset Modality Dominance\n'
                  '(% samples: Visual > Metadata confidence)',
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Percentage of Samples (%)', fontsize=11)
axes[1].set_xticklabels(datasets_mc, rotation=25, ha='right', fontsize=10)
axes[1].legend(['Visual Dominant','Metadata Dominant'], fontsize=10)
axes[1].yaxis.grid(True, linestyle='--', alpha=0.6); axes[1].set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'Modality_Contribution.png'),
            dpi=300, bbox_inches='tight')
plt.show()

# ── Panel 3: Per-dataset KDE distributions ───────────────────
n_ds = len(datasets_mc)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, ds in enumerate(datasets_mc):
    sub = val_analysis[val_analysis['dataset']==ds]
    sub['Visual_Prob'].plot.kde(ax=axes[i], color='#4C72B0', lw=2.5, label='Visual')
    sub['Meta_Prob'].plot.kde(ax=axes[i], color='#DD8452', lw=2.5,
                              linestyle='--', label='Metadata')
    axes[i].set_title(ds, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Prediction Probability', fontsize=10)
    axes[i].set_ylabel('Density', fontsize=10)
    axes[i].set_xlim(0, 1)
    axes[i].legend(fontsize=9)
    axes[i].grid(True, linestyle='--', alpha=0.5)
plt.suptitle('Visual vs. Metadata Confidence Distribution per Dataset',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'Modality_KDE_per_Dataset.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 20. Synergy Analysis - Best-of-Both Oracle
Formal test of fusion synergy: the oracle accuracy is the upper bound achievable if a perfect modality selector always chose the correct stream when at least one baseline was right. Comparing our fusion model against this ceiling quantifies the practical benefit of learned cross-modal interaction.


In [ ]:
vis_ok  = (y_pred_vis_abl == y_true_np)
meta_ok = (y_pred_met_abl == y_true_np)
fus_ok  = (y_pred_fus_abl == y_true_np)

oracle_ok  = vis_ok | meta_ok
oracle_acc = oracle_ok.mean() * 100
fusion_acc_syn = fus_ok.mean() * 100

both_wrong       = (~vis_ok) & (~meta_ok)
fus_recovered    = fus_ok[both_wrong].sum()
recovery_rate    = 100 * fus_recovered / max(both_wrong.sum(), 1)

uncertain_vis    = (y_prob_vis_abl >= 0.4) & (y_prob_vis_abl <= 0.6)
uncertain_meta   = (y_prob_met_abl >= 0.4) & (y_prob_met_abl <= 0.6)
both_uncertain   = uncertain_vis & uncertain_meta
fus_resolves     = fus_ok[both_uncertain].sum()
resolution_rate  = 100 * fus_resolves / max(both_uncertain.sum(), 1)

print("=" * 60)
print("  SYNERGY ANALYSIS")
print("=" * 60)
print(f"  Visual-Only Accuracy:       {vis_ok.mean()*100:.2f}%")
print(f"  Metadata-Only Accuracy:     {meta_ok.mean()*100:.2f}%")
print(f"  Oracle (Best-of-Both):      {oracle_acc:.2f}%")
print(f"  Proposed Fusion:            {fusion_acc_syn:.2f}%")
print(f"  Gap (Oracle - Fusion):      {oracle_acc - fusion_acc_syn:.2f} pp")
print(f"\n  Both baselines wrong:       {both_wrong.sum()}")
print(f"  Fusion recovered:           {fus_recovered} ({recovery_rate:.1f}%)")
print(f"\n  Both baselines uncertain:   {both_uncertain.sum()}")
print(f"  Fusion correctly resolved:  {fus_resolves} ({resolution_rate:.1f}%)")

# ── Visualise ─────────────────────────────────────────────────
labels_syn = ['Visual\nOnly','Metadata\nOnly','Average\nFusion',
              'Proposed\nFusion','Oracle\n(Upper Bound)']
accs_syn   = [vis_ok.mean()*100, meta_ok.mean()*100,
              ((y_prob_avg>0.5)==y_true_np).mean()*100,
              fusion_acc_syn, oracle_acc]
colors_syn = ['#4C72B0','#DD8452','#8172B2','#55A868','#BCB9B9']

fig, ax = plt.subplots(figsize=(11,5))
bars = ax.bar(labels_syn, accs_syn, color=colors_syn, edgecolor='white', width=0.55)
ax.axhline(oracle_acc, color='grey', linestyle='--', lw=1.5,
           label=f'Oracle ceiling: {oracle_acc:.1f}%')
for bar, v in zip(bars, accs_syn):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 112)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Synergy Analysis: Proposed Fusion vs. Best-of-Both Oracle',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.6); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'Synergy_Oracle.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 21. Zero-Metadata Scenario Test
Simulating complete EXIF stripping (as occurs on WhatsApp, Facebook, Twitter) by zeroing all metadata features. This establishes the performance floor of the framework when operating purely on visual evidence.


In [ ]:
X_zero_meta    = np.zeros_like(X_meta_val_aligned)
X_fusion_zero  = np.hstack([X_visual_val, X_zero_meta])

y_prob_zero = xgb_fusion.predict_proba(X_fusion_zero)[:, 1]
y_pred_zero = (y_prob_zero > 0.5).astype(int)

acc_full = accuracy_score(y_true_np, y_pred_fusion)
acc_zero = accuracy_score(y_true_np, y_pred_zero)
f1_zero  = f1_score(y_true_np, y_pred_zero, zero_division=0)
auc_zero = roc_auc_score(y_true_np, y_prob_zero)

print("=" * 60)
print("  ZERO-METADATA SCENARIO (Complete EXIF Strip)")
print("=" * 60)
print(f"  Full metadata — Accuracy: {acc_full*100:.2f}%")
print(f"  Zero metadata — Accuracy: {acc_zero*100:.2f}%  "
      f"F1: {f1_zero*100:.2f}%  AUC: {auc_zero:.4f}")
print(f"  Performance drop: {(acc_full-acc_zero)*100:.2f} percentage points")

zm_df = df[df['split']=='VALIDATION'].copy().reset_index(drop=True)
zm_df['Full_Pred'] = y_pred_fusion
zm_df['Zero_Pred'] = y_pred_zero

print("\n  Per-Dataset Accuracy:")
print(f"  {'Dataset':<12} | {'Full':>8} | {'Zero':>8} | {'Drop':>8}")
print(f"  {'-'*44}")
ds_accs_full, ds_accs_zero, ds_labels = [], [], []
for ds in sorted(zm_df['dataset'].unique()):
    sub = zm_df[zm_df['dataset']==ds]
    af  = accuracy_score(sub['label'], sub['Full_Pred'])
    az  = accuracy_score(sub['label'], sub['Zero_Pred'])
    print(f"  {ds:<12} | {af*100:>8.2f} | {az*100:>8.2f} | {(af-az)*100:>8.2f}")
    ds_accs_full.append(af*100); ds_accs_zero.append(az*100); ds_labels.append(ds)

xz = np.arange(len(ds_labels))
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(xz-0.2, ds_accs_full, 0.38, label='Full Metadata',
       color='#4C72B0', edgecolor='white')
ax.bar(xz+0.2, ds_accs_zero, 0.38, label='Zero Metadata (Stripped)',
       color='#C44E52', edgecolor='white', alpha=0.85)
ax.set_xticks(xz); ax.set_xticklabels(ds_labels, fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_ylim(0, 112)
ax.set_title('Full Metadata vs. Zero-Metadata (Stripped) — Per Dataset',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.6); ax.set_axisbelow(True)
for i,(vf,vz) in enumerate(zip(ds_accs_full, ds_accs_zero)):
    ax.text(i-0.2, vf+1, f'{vf:.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.text(i+0.2, vz+1, f'{vz:.1f}%', ha='center', fontsize=9,
            fontweight='bold', color='#C44E52')
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'Zero_Metadata_Test.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 22. 5-Fold Stratified Cross-Validation
Applied to the full fusion feature space (visual embeddings + metadata) to verify statistical stability of the reported results. Mean and standard deviation reported across all folds.


In [ ]:
N_FOLDS  = 5
skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=Config.SEED)

X_all_fusion = np.vstack([X_fusion_train, X_fusion_val])
X_all_meta   = np.vstack([X_meta_train_aligned, X_meta_val_aligned])
y_all        = pd.concat([y_train, y_val], ignore_index=True).values

print(f"5-Fold CV on {len(y_all):,} samples  |  "
      f"Feature shape: {X_all_fusion.shape}\n")

cv_fusion, cv_meta = [], []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_all_fusion, y_all)):
    X_tr_f, X_vl_f = X_all_fusion[tr_idx], X_all_fusion[vl_idx]
    X_tr_m, X_vl_m = X_all_meta[tr_idx],   X_all_meta[vl_idx]
    y_tr, y_vl     = y_all[tr_idx],          y_all[vl_idx]

    # Fusion model
    fold_fus = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05,
                                   max_depth=6, subsample=0.8, colsample_bytree=0.8,
                                   use_label_encoder=False, eval_metric='logloss',
                                   random_state=Config.SEED)
    fold_fus.fit(X_tr_f, y_tr, verbose=False)
    yp = fold_fus.predict_proba(X_vl_f)[:,1]
    yd = (yp>0.5).astype(int)
    cv_fusion.append({'Accuracy': accuracy_score(y_vl,yd),
                      'Precision': precision_score(y_vl,yd,zero_division=0),
                      'Recall': recall_score(y_vl,yd,zero_division=0),
                      'F1': f1_score(y_vl,yd,zero_division=0),
                      'AUC-ROC': roc_auc_score(y_vl,yp)})

    # Metadata-only model
    fold_met = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5,
                                   use_label_encoder=False, eval_metric='logloss',
                                   random_state=Config.SEED)
    fold_met.fit(X_tr_m, y_tr, verbose=False)
    ypm = fold_met.predict_proba(X_vl_m)[:,1]
    ydm = (ypm>0.5).astype(int)
    cv_meta.append({'Accuracy': accuracy_score(y_vl,ydm),
                    'Precision': precision_score(y_vl,ydm,zero_division=0),
                    'Recall': recall_score(y_vl,ydm,zero_division=0),
                    'F1': f1_score(y_vl,ydm,zero_division=0),
                    'AUC-ROC': roc_auc_score(y_vl,ypm)})
    print(f"  Fold {fold+1}/{N_FOLDS} — Fusion: Acc={cv_fusion[-1]['Accuracy']:.4f} "
          f"F1={cv_fusion[-1]['F1']:.4f}  |  "
          f"Meta: Acc={cv_meta[-1]['Accuracy']:.4f}")

# Summary
METRICS = ['Accuracy','Precision','Recall','F1','AUC-ROC']
cv_rows = []
for model_name, fold_list in [('Feature-Level Fusion', cv_fusion),
                                ('Metadata-Only',        cv_meta)]:
    for m in METRICS:
        vals = [f[m] for f in fold_list]
        cv_rows.append({'Model': model_name, 'Metric': m,
                        'Mean': np.mean(vals), 'Std': np.std(vals)})

cv_summary = pd.DataFrame(cv_rows)
print("\n5-Fold CV Summary:")
for model_name in ['Feature-Level Fusion','Metadata-Only']:
    sub = cv_summary[cv_summary['Model']==model_name]
    print(f"\n  {model_name}")
    print(f"  {'Metric':<12} {'Mean':>8} {'Std':>8}")
    for _, r in sub.iterrows():
        print(f"  {r['Metric']:<12} {r['Mean']:>8.4f} {r['Std']:>8.4f}")

# ── Bar chart ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
for ax, model_name, color in zip(axes,
        ['Feature-Level Fusion','Metadata-Only'],
        ['#4C72B0','#DD8452']):
    sub = cv_summary[cv_summary['Model']==model_name]
    bars = ax.bar(sub['Metric'], sub['Mean'], yerr=sub['Std'],
                  color=color, edgecolor='white', width=0.55, capsize=6,
                  error_kw=dict(elinewidth=2, ecolor='#222'))
    for bar, mean, std in zip(bars, sub['Mean'], sub['Std']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+0.005,
                f'{mean:.3f}\n±{std:.3f}', ha='center', va='bottom', fontsize=8.5,
                fontweight='bold')
    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.set_ylabel('Score', fontsize=11)
    ax.set_ylim(0, 1.18)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6); ax.set_axisbelow(True)
plt.suptitle(f'5-Fold Stratified Cross-Validation (Mean ± Std, N={N_FOLDS} folds)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_DIR,'CV_5Fold_Results.png'),
            dpi=300, bbox_inches='tight')
plt.show()